# Incremental training and checkpoint recovery

Historical research source, original zero-based cells **24?30**. Outputs have been cleared.

These cells retain the original global state, Colab paths, manual annotations, and checkpoint assumptions. This notebook is not standalone; earlier state and external artifacts may be required. Some repair branches are alternatives. Read [the migration guide](../../docs/migration.md) before selecting cells to run. For a new baseline, use [the package workflow](../amharic_asr.ipynb).


## Original source cell 24

Position 25 in the original notebook; original code comments may use different numbering.


In [ ]:

import math
import json
import random
import shutil
import gc
from pathlib import Path

import torch
from transformers import get_cosine_schedule_with_warmup

# ==========================================
# 1. EXPERIMENT CONFIGURATION
# ==========================================

TRAINING_SEED = 42

NUM_EPOCHS = 3
GRAD_ACCUM_STEPS = 4
STEPS_THIS_RUN = 10

ENCODER_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

assert len(train_dataset) == 5083

STEPS_PER_EPOCH = math.ceil(
    len(train_dataset) / GRAD_ACCUM_STEPS
)

TOTAL_STEPS = NUM_EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = math.ceil(TOTAL_STEPS * 0.05)

CHECKPOINT_NAME = f"step_{STEPS_THIS_RUN:06d}"
SAVE_DIR = CHECKPOINT_DIR / CHECKPOINT_NAME

assert not SAVE_DIR.exists(), (
    f"Checkpoint already exists: {SAVE_DIR}\n"
    "Do not restart from step zero over an existing checkpoint."
)

torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)

# ==========================================
# 2. CREATE REPRODUCIBLE TRAINING ORDER
# ==========================================

training_order = list(range(len(train_dataset)))

rng = random.Random(TRAINING_SEED)
rng.shuffle(training_order)

# First 40 recordings in the deterministic order.
num_records = STEPS_THIS_RUN * GRAD_ACCUM_STEPS

selected_indices = training_order[:num_records]

assert len(selected_indices) == 40

# ==========================================
# 3. OPTIMIZER AND LEARNING-RATE SCHEDULER
# ==========================================

encoder_params = [
    p
    for layer in model.wav2vec2.encoder.layers[-2:]
    for p in layer.parameters()
    if p.requires_grad
]

head_params = [
    p for p in model.lm_head.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)

scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0
)

model.train()
model.zero_grad(set_to_none=True)

torch.cuda.reset_peak_memory_stats()

print("TRAINING CONFIGURATION")
print("-" * 50)
print("Training recordings:", len(train_dataset))
print("Planned epochs:", NUM_EPOCHS)
print("Updates per epoch:", STEPS_PER_EPOCH)
print("Total planned updates:", TOTAL_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Updates in this run:", STEPS_THIS_RUN)

# ==========================================
# 4. RUN FIRST 10 OPTIMIZER UPDATES
# ==========================================

step_losses = []
successful_updates = 0

for update_index in range(STEPS_THIS_RUN):

    optimizer.zero_grad(set_to_none=True)

    micro_losses = []

    start = update_index * GRAD_ACCUM_STEPS
    end = start + GRAD_ACCUM_STEPS

    for record_index in selected_indices[start:end]:

        batch = ctc_collator(
            [train_dataset[record_index]]
        )

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        assert torch.isfinite(loss).item(), (
            "Non-finite training loss encountered."
        )

        micro_losses.append(float(loss.item()))

        scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        del batch, outputs, loss, scaled_loss

    # Unscale before gradient clipping.
    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        encoder_params + head_params,
        max_norm=MAX_GRAD_NORM,
        error_if_nonfinite=True
    )

    previous_scale = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    current_scale = scaler.get_scale()

    # A scale reduction indicates an AMP overflow
    # caused the optimizer update to be skipped.
    if current_scale < previous_scale:
        raise RuntimeError(
            "AMP skipped an optimizer update. "
            "Stop and investigate before checkpointing."
        )

    scheduler.step()
    successful_updates += 1

    mean_loss = sum(micro_losses) / len(micro_losses)
    step_losses.append(mean_loss)

    current_encoder_lr = optimizer.param_groups[0]["lr"]
    current_head_lr = optimizer.param_groups[1]["lr"]

    print(
        f"Update {successful_updates:02d}/{STEPS_THIS_RUN} | "
        f"CTC loss: {mean_loss:.4f} | "
        f"Grad norm: {float(grad_norm):.3f} | "
        f"Head LR: {current_head_lr:.8f}"
    )

assert successful_updates == STEPS_THIS_RUN

# ==========================================
# 5. SAVE COMPLETE RESUMABLE CHECKPOINT
# ==========================================

LOCAL_SAVE_DIR = Path(
    f"/content/asr_{CHECKPOINT_NAME}"
)

LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=False)

# Save trained model weights and processor.
model.save_pretrained(
    str(LOCAL_SAVE_DIR),
    safe_serialization=True
)

processor.save_pretrained(str(LOCAL_SAVE_DIR))

# Save optimizer, scheduler, AMP scaler and RNG state.
trainer_state = {
    "global_step": successful_updates,
    "epoch": 0,
    "next_sample_offset": num_records,
    "seed": TRAINING_SEED,
    "num_epochs": NUM_EPOCHS,
    "gradient_accumulation": GRAD_ACCUM_STEPS,
    "total_steps": TOTAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "unfrozen_layers": 2,
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),
    "torch_rng_state": torch.get_rng_state(),
    "cuda_rng_states": torch.cuda.get_rng_state_all()
}

torch.save(
    trainer_state,
    LOCAL_SAVE_DIR / "trainer_state.pt"
)

training_summary = {
    "training_step": successful_updates,
    "recordings_processed": num_records,
    "mean_ctc_loss": sum(step_losses) / len(step_losses),
    "step_losses": step_losses,
    "planned_epochs": NUM_EPOCHS,
    "dataset_recordings": len(train_dataset)
}

with (LOCAL_SAVE_DIR / "training_summary.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(training_summary, file, indent=2)

# Copy checkpoint to persistent storage.
shutil.copytree(LOCAL_SAVE_DIR, SAVE_DIR)

# Validate copied file sizes before marking complete.
for source in LOCAL_SAVE_DIR.iterdir():
    if source.is_file():
        destination = SAVE_DIR / source.name

        assert destination.exists()
        assert destination.stat().st_size == source.stat().st_size

(SAVE_DIR / "COMPLETE").write_text(
    "Checkpoint saved successfully.",
    encoding="utf-8"
)

# Remove temporary local checkpoint copy.
shutil.rmtree(LOCAL_SAVE_DIR)

# ==========================================
# 6. TRAINING SUMMARY
# ==========================================

print("\n" + "=" * 55)
print("FIRST TRAINING CHUNK COMPLETE")
print("=" * 55)

print("Successful optimizer updates:", successful_updates)
print("Recordings processed:", num_records)
print("Mean CTC loss:", round(training_summary["mean_ctc_loss"], 4))

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print("Checkpoint:", SAVE_DIR)
print("Checkpoint complete:", (SAVE_DIR / "COMPLETE").exists())
print("Starting baseline remains untouched.")

print("\nREADY FOR CONTINUED TRAINING")


## Original source cell 25

Position 26 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import random
import time
import shutil
import hashlib
import os
from pathlib import Path

import torch
import numpy as np

# ==========================================
# 1. CONTINUATION CONFIGURATION
# ==========================================

START_STEP = 10
END_STEP = 50
ACCUMULATION = 4
SEED = 42

assert len(train_dataset) == 5083
assert successful_updates == START_STEP
assert scheduler.last_epoch == START_STEP
assert model.is_gradient_checkpointing

SAVE_DIR = CHECKPOINT_DIR / f"step_{END_STEP:06d}"
PARTIAL_DIR = CHECKPOINT_DIR / f"step_{END_STEP:06d}.partial"

assert not SAVE_DIR.exists(), (
    f"Checkpoint already exists: {SAVE_DIR}"
)
assert not PARTIAL_DIR.exists(), (
    f"Incomplete checkpoint exists: {PARTIAL_DIR}"
)

# ==========================================
# 2. RECONSTRUCT AND SAVE TRAINING ORDER
# ==========================================

training_order = list(range(len(train_dataset)))
random.Random(SEED).shuffle(training_order)

ORDER_PATH = DATA_DIR / "training_order_25h.json"

order_record = {
    "seed": SEED,
    "recording_count": len(train_dataset),
    "recording_ids": [
        train_records[i]["id"]
        for i in training_order
    ]
}

if ORDER_PATH.exists():
    with ORDER_PATH.open(encoding="utf-8") as f:
        previous_order = json.load(f)

    assert previous_order == order_record, (
        "Training order changed. Cannot safely resume."
    )
else:
    with ORDER_PATH.open("w", encoding="utf-8") as f:
        json.dump(order_record, f, ensure_ascii=False)

print("CONTINUING TRAINING")
print("-" * 50)
print("Starting step:", START_STEP)
print("Ending step:", END_STEP)
print("Additional recordings:", (END_STEP - START_STEP) * 4)
print("Training order verified:", True)

# ==========================================
# 3. TRAIN THE NEXT 40 UPDATES
# ==========================================

model.train()
torch.cuda.reset_peak_memory_stats()

training_log = []
start_time = time.perf_counter()

for step in range(START_STEP + 1, END_STEP + 1):

    optimizer.zero_grad(set_to_none=True)

    # Continue exactly where step 10 stopped.
    offset = (step - 1) * ACCUMULATION

    indices = training_order[
        offset:offset + ACCUMULATION
    ]

    assert len(indices) == ACCUMULATION

    losses = []

    for record_index in indices:

        batch = ctc_collator([
            train_dataset[record_index]
        ])

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        assert torch.isfinite(loss).item(), (
            f"Non-finite loss at step {step}"
        )

        losses.append(float(loss.item()))

        scaler.scale(
            loss / ACCUMULATION
        ).backward()

        del batch, outputs, loss

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        encoder_params + head_params,
        max_norm=MAX_GRAD_NORM,
        error_if_nonfinite=True
    )

    previous_scale = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < previous_scale:
        raise RuntimeError(
            f"AMP skipped optimizer update at step {step}. "
            "Resume from the last saved checkpoint."
        )

    scheduler.step()
    successful_updates = step

    mean_loss = sum(losses) / len(losses)

    training_log.append({
        "step": step,
        "loss": mean_loss,
        "gradient_norm": float(grad_norm),
        "encoder_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"]
    })

    if step % 5 == 0:
        recent = training_log[-5:]

        print(
            f"Step {step:04d}/{TOTAL_STEPS} | "
            f"Last-5 loss: "
            f"{np.mean([x['loss'] for x in recent]):.4f} | "
            f"Head LR: {optimizer.param_groups[1]['lr']:.8f} | "
            f"Grad norm: {float(grad_norm):.3f}"
        )

training_seconds = time.perf_counter() - start_time

assert successful_updates == END_STEP

# ==========================================
# 4. SAVE COMPLETE TRAINING STATE
# ==========================================

LOCAL_SAVE_DIR = Path(
    f"/content/asr_checkpoint_step_{END_STEP:06d}"
)

assert not LOCAL_SAVE_DIR.exists()
LOCAL_SAVE_DIR.mkdir(parents=True)

model.save_pretrained(
    str(LOCAL_SAVE_DIR),
    safe_serialization=True
)

processor.save_pretrained(str(LOCAL_SAVE_DIR))

state = {
    "global_step": END_STEP,
    "epoch": 0,
    "next_sample_offset": END_STEP * ACCUMULATION,
    "seed": SEED,
    "num_epochs": NUM_EPOCHS,
    "gradient_accumulation": ACCUMULATION,
    "total_steps": TOTAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "unfrozen_layers": 2,
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),
    "torch_rng_state": torch.get_rng_state(),
    "cuda_rng_states": torch.cuda.get_rng_state_all()
}

torch.save(
    state,
    LOCAL_SAVE_DIR / "trainer_state.pt"
)

summary = {
    "starting_step": START_STEP,
    "ending_step": END_STEP,
    "recordings_processed_total": END_STEP * ACCUMULATION,
    "chunk_seconds": training_seconds,
    "chunk_mean_loss": float(np.mean([
        x["loss"] for x in training_log
    ])),
    "training_log": training_log
}

with (LOCAL_SAVE_DIR / "training_summary.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

# ==========================================
# 5. BACK UP AND VERIFY CHECKPOINT
# ==========================================

def file_sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024**2),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

shutil.copytree(LOCAL_SAVE_DIR, PARTIAL_DIR)

for original in LOCAL_SAVE_DIR.iterdir():
    if not original.is_file():
        continue

    copied = PARTIAL_DIR / original.name

    assert copied.exists()
    assert file_sha256(original) == file_sha256(copied), (
        f"Checkpoint verification failed: {original.name}"
    )

# Rename verified checkpoint into its final location.
os.rename(PARTIAL_DIR, SAVE_DIR)

(SAVE_DIR / "COMPLETE").write_text(
    "Checkpoint complete and verified.",
    encoding="utf-8"
)

shutil.rmtree(LOCAL_SAVE_DIR)

# ==========================================
# 6. REPORT
# ==========================================

print("\n" + "=" * 55)
print("STEP 50 CHECKPOINT COMPLETE")
print("=" * 55)

print("Global optimizer step:", successful_updates)
print("Recordings processed:", END_STEP * ACCUMULATION)
print("Mean loss for this chunk:", round(summary["chunk_mean_loss"], 4))
print("Training time (minutes):", round(training_seconds / 60, 2))

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

print("Checkpoint:", SAVE_DIR)
print("Checkpoint verified:", (SAVE_DIR / "COMPLETE").exists())
print("Training order:", ORDER_PATH)

print("\nREADY FOR STEP-50 EVALUATION")


## Original source cell 26

Position 27 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import shutil
import numpy as np
import torch

from pathlib import Path
from torchcodec.decoders import AudioDecoder
from jiwer import cer, wer
from IPython.display import Audio, display
from tqdm.auto import tqdm

# ==========================================
# 1. VERIFY CHECKPOINT AND BASELINE
# ==========================================

EVAL_STEP = 50

assert successful_updates == EVAL_STEP
assert (
    CHECKPOINT_DIR / "step_000050" / "COMPLETE"
).exists(), "Step-50 checkpoint is missing."

baseline_metrics_path = (
    DATA_DIR / "baseline_step0_metrics.json"
)

baseline_predictions_path = (
    DATA_DIR / "baseline_step0_predictions_12.jsonl"
)

with baseline_metrics_path.open(encoding="utf-8") as f:
    baseline_metrics = json.load(f)

with baseline_predictions_path.open(encoding="utf-8") as f:
    baseline_records = [
        json.loads(line) for line in f
    ]

baseline_by_id = {
    record["id"]: record
    for record in baseline_records
}

with BASELINE_EVAL_PATH.open(encoding="utf-8") as f:
    eval_records = [
        json.loads(line) for line in f
    ]

assert len(eval_records) == 12
assert {r["id"] for r in eval_records} == set(baseline_by_id)

print("Evaluation step:", EVAL_STEP)
print("Fixed validation recordings:", len(eval_records))
print("Baseline loaded: YES")

# ==========================================
# 2. VERIFY EVALUATION AUDIO
# ==========================================

EVAL_AUDIO_DIR = Path("/content/amharic_eval_audio")
EVAL_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_EVAL_AUDIO = PROJECT_DIR / "evaluation_audio"

for record in eval_records:
    filename = record["id"] + ".mp3"

    local_path = EVAL_AUDIO_DIR / filename
    drive_path = DRIVE_EVAL_AUDIO / filename

    if not local_path.exists():
        assert drive_path.exists(), (
            f"Evaluation audio missing: {filename}"
        )

        shutil.copy2(drive_path, local_path)

# ==========================================
# 3. INFERENCE WITH STEP-50 MODEL
# ==========================================

model.eval()

results = []
references = []
predictions = []
preview_audio = None

try:
    with torch.inference_mode():
        for i, record in enumerate(
            tqdm(eval_records, desc="Step-50 evaluation")
        ):
            path = EVAL_AUDIO_DIR / (
                record["id"] + ".mp3"
            )

            decoder = AudioDecoder(
                str(path),
                sample_rate=SAMPLE_RATE,
                num_channels=1
            )

            decoded = decoder.get_all_samples()

            waveform = (
                decoded.data[0]
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            # Retain one clip for listening after evaluation.
            if i == 0:
                preview_audio = waveform.copy()

            inputs = processor.feature_extractor(
                waveform,
                sampling_rate=SAMPLE_RATE,
                return_attention_mask=True,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(DEVICE)
                for key, value in inputs.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(**inputs).logits

            assert torch.isfinite(logits).all(), (
                f"Non-finite logits for {record['id']}"
            )

            token_ids = logits.argmax(dim=-1)

            predicted = processor.batch_decode(
                token_ids.cpu()
            )[0]

            predicted = normalize_amharic_ctc(
                predicted
            )

            reference = record["transcription"]

            blank_fraction = (
                (token_ids == tokenizer.pad_token_id)
                .float()
                .mean()
                .item()
            )

            result = {
                "id": record["id"],
                "speaker_id": record["speaker_id"],
                "reference": reference,
                "prediction": predicted,
                "duration_seconds": len(waveform) / SAMPLE_RATE,
                "cer": cer(reference, predicted),
                "wer": wer(reference, predicted),
                "blank_fraction": blank_fraction,
                "prediction_length": len(predicted)
            }

            results.append(result)
            references.append(reference)
            predictions.append(predicted)

            del inputs, logits, token_ids

finally:
    # Restore training mode without changing weights.
    model.train()

# ==========================================
# 4. COMPUTE COMPARABLE METRICS
# ==========================================

step50_cer = cer(references, predictions)
step50_wer = wer(references, predictions)

mean_blank = float(np.mean([
    r["blank_fraction"] for r in results
]))

empty_count = sum(
    not prediction for prediction in predictions
)

mean_prediction_length = float(np.mean([
    len(prediction) for prediction in predictions
]))

metrics = {
    "training_step": EVAL_STEP,
    "evaluation_recordings": len(results),
    "cer": step50_cer,
    "wer": step50_wer,
    "mean_blank_fraction": mean_blank,
    "empty_predictions": empty_count,
    "mean_prediction_length": mean_prediction_length
}

# ==========================================
# 5. SAVE RESULTS TO GOOGLE DRIVE
# ==========================================

RESULTS_PATH = (
    DATA_DIR / "eval_step_000050_predictions_12.jsonl"
)

METRICS_PATH = (
    DATA_DIR / "eval_step_000050_metrics.json"
)

with RESULTS_PATH.open("w", encoding="utf-8") as f:
    for result in results:
        f.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# ==========================================
# 6. BASELINE COMPARISON
# ==========================================

print("\n" + "=" * 65)
print("BEFORE VS AFTER: STEP 0 AND STEP 50")
print("=" * 65)

print(
    f"{'Metric':27}"
    f"{'Step 0':>15}"
    f"{'Step 50':>15}"
)

print("-" * 57)

print(
    f"{'CER':27}"
    f"{baseline_metrics['cer'] * 100:14.2f}%"
    f"{step50_cer * 100:14.2f}%"
)

print(
    f"{'WER':27}"
    f"{baseline_metrics['wer'] * 100:14.2f}%"
    f"{step50_wer * 100:14.2f}%"
)

print(
    f"{'Mean blank frames':27}"
    f"{baseline_metrics['mean_blank_fraction'] * 100:14.2f}%"
    f"{mean_blank * 100:14.2f}%"
)

print(
    f"{'Empty predictions':27}"
    f"{baseline_metrics['empty_predictions']:15}"
    f"{empty_count:15}"
)

print("\nCER change (percentage points):",
      round(
          (step50_cer - baseline_metrics["cer"]) * 100,
          2
      ))

# ==========================================
# 7. DISPLAY SIDE-BY-SIDE TRANSCRIPTIONS
# ==========================================

print("\nAMHARIC TRANSCRIPTION COMPARISON")

for i in [0, 5, 11]:
    current = results[i]
    original = baseline_by_id[current["id"]]

    print("\n" + "-" * 65)
    print("RECORDING:", i + 1)
    print("REFERENCE:")
    print(current["reference"][:220])

    print("\nSTEP 0:")
    print(original["prediction"][:220] or "[EMPTY]")

    print("\nSTEP 50:")
    print(current["prediction"][:220] or "[EMPTY]")

    print(
        "\nCER:",
        f"{original['cer'] * 100:.1f}%",
        "->",
        f"{current['cer'] * 100:.1f}%"
    )

print("\nLISTEN TO RECORDING 1:")
display(Audio(preview_audio, rate=SAMPLE_RATE))

print("\nResults saved:", RESULTS_PATH)
print("Metrics saved:", METRICS_PATH)
print("\nSTEP-50 EVALUATION COMPLETE")


## Original source cell 27

Position 28 in the original notebook; original code comments may use different numbering.


In [ ]:

import os
import json
import time
import shutil
import hashlib
import random
from pathlib import Path

import numpy as np
import torch

# ==========================================
# 1. CONTINUATION SETTINGS
# ==========================================

START_STEP = 50
END_STEP = 200
ACCUMULATION = 4

assert successful_updates == START_STEP
assert scheduler.last_epoch == START_STEP
assert model.training
assert model.is_gradient_checkpointing

# Reconstruct the saved deterministic order.
training_order = list(range(len(train_dataset)))
random.Random(TRAINING_SEED).shuffle(training_order)

with ORDER_PATH.open(encoding="utf-8") as f:
    saved_order = json.load(f)

assert [
    train_records[i]["id"]
    for i in training_order
] == saved_order["recording_ids"], (
    "Training order mismatch."
)

assert END_STEP * ACCUMULATION <= len(training_order)

checkpoint_path = (
    CHECKPOINT_DIR / f"step_{END_STEP:06d}"
)

partial_path = (
    CHECKPOINT_DIR / f"step_{END_STEP:06d}.partial"
)

local_path = Path(
    f"/content/asr_step_{END_STEP:06d}_temp"
)

for path in [checkpoint_path, partial_path, local_path]:
    assert not path.exists(), (
        f"Existing checkpoint path: {path}"
    )

print("TRAINING CONTINUATION")
print("-" * 50)
print("Starting step:", START_STEP)
print("Target step:", END_STEP)
print(
    "Additional recordings:",
    (END_STEP - START_STEP) * ACCUMULATION
)
print("Warmup finishes at step:", WARMUP_STEPS)
print("Training order: VERIFIED")

# ==========================================
# 2. CONTINUE OPTIMIZER UPDATES
# ==========================================

model.train()
torch.cuda.reset_peak_memory_stats()

training_log = []
start_time = time.perf_counter()

for step in range(START_STEP + 1, END_STEP + 1):

    optimizer.zero_grad(set_to_none=True)

    offset = (step - 1) * ACCUMULATION
    indices = training_order[offset:offset + ACCUMULATION]

    assert len(indices) == ACCUMULATION

    losses = []

    for record_index in indices:

        batch = ctc_collator(
            [train_dataset[record_index]]
        )

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at step {step}"
            )

        losses.append(float(loss.item()))

        scaler.scale(
            loss / ACCUMULATION
        ).backward()

        del batch, outputs, loss

    # Unscale before clipping.
    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        encoder_params + head_params,
        max_norm=MAX_GRAD_NORM,
        error_if_nonfinite=True
    )

    old_scale = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < old_scale:
        raise RuntimeError(
            f"AMP skipped update at step {step}. "
            "Reload the last verified checkpoint."
        )

    scheduler.step()
    successful_updates = step

    mean_loss = float(np.mean(losses))

    training_log.append({
        "step": step,
        "loss": mean_loss,
        "gradient_norm": float(grad_norm),
        "encoder_lr": optimizer.param_groups[0]["lr"],
        "head_lr": optimizer.param_groups[1]["lr"]
    })

    if step % 25 == 0:
        recent_losses = [
            entry["loss"]
            for entry in training_log[-25:]
        ]

        print(
            f"Step {step:04d}/{TOTAL_STEPS} | "
            f"Mean loss (last 25): "
            f"{np.mean(recent_losses):.4f} | "
            f"Head LR: "
            f"{optimizer.param_groups[1]['lr']:.8f}"
        )

elapsed_seconds = time.perf_counter() - start_time

assert successful_updates == END_STEP

# ==========================================
# 3. SAVE MODEL + COMPLETE TRAINING STATE
# ==========================================

local_path.mkdir(parents=True)

model.save_pretrained(
    str(local_path),
    safe_serialization=True
)

processor.save_pretrained(str(local_path))

trainer_state = {
    "global_step": END_STEP,
    "epoch": 0,
    "next_sample_offset": END_STEP * ACCUMULATION,
    "seed": TRAINING_SEED,
    "num_epochs": NUM_EPOCHS,
    "gradient_accumulation": ACCUMULATION,
    "total_steps": TOTAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "unfrozen_layers": 2,
    "optimizer": optimizer.state_dict(),
    "scheduler": scheduler.state_dict(),
    "scaler": scaler.state_dict(),
    "torch_rng_state": torch.get_rng_state(),
    "cuda_rng_states": torch.cuda.get_rng_state_all()
}

torch.save(
    trainer_state,
    local_path / "trainer_state.pt"
)

summary = {
    "starting_step": START_STEP,
    "ending_step": END_STEP,
    "total_recordings_processed": END_STEP * ACCUMULATION,
    "mean_chunk_loss": float(np.mean([
        entry["loss"] for entry in training_log
    ])),
    "elapsed_seconds": elapsed_seconds,
    "training_log": training_log
}

with (local_path / "training_summary.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

# ==========================================
# 4. VERIFIED GOOGLE DRIVE BACKUP
# ==========================================

def checkpoint_hash(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024**2),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

shutil.copytree(local_path, partial_path)

for source in local_path.iterdir():
    if source.is_file():
        destination = partial_path / source.name

        assert destination.exists()
        assert checkpoint_hash(source) == checkpoint_hash(destination), (
            f"Checksum mismatch: {source.name}"
        )

os.rename(partial_path, checkpoint_path)

(checkpoint_path / "COMPLETE").write_text(
    "Verified checkpoint.",
    encoding="utf-8"
)

shutil.rmtree(local_path)

# ==========================================
# 5. REPORT
# ==========================================

print("\n" + "=" * 55)
print("STEP 200 TRAINING COMPLETE")
print("=" * 55)

print("Global optimizer step:", successful_updates)
print("Recordings processed:", END_STEP * ACCUMULATION)
print("Mean chunk loss:", round(summary["mean_chunk_loss"], 4))
print("Training duration (minutes):", round(elapsed_seconds / 60, 2))

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

print("Current head LR:", optimizer.param_groups[1]["lr"])
print("Warmup completed:", END_STEP >= WARMUP_STEPS)
print("Checkpoint:", checkpoint_path)
print(
    "Checkpoint verified:",
    (checkpoint_path / "COMPLETE").exists()
)

print("\nREADY FOR STEP-200 EVALUATION")


## Original source cell 28

Position 29 in the original notebook; original code comments may use different numbering.


In [ ]:

import json
import numpy as np
import torch

from pathlib import Path
from torchcodec.decoders import AudioDecoder
from jiwer import cer, wer
from tqdm.auto import tqdm
from IPython.display import Audio, display

# ==========================================
# 1. VERIFY EXPERIMENT STATE
# ==========================================

EVAL_STEP = 200

assert successful_updates == EVAL_STEP
assert (
    CHECKPOINT_DIR / "step_000200" / "COMPLETE"
).exists()

# Load both previous evaluations.
with (
    DATA_DIR / "baseline_step0_metrics.json"
).open(encoding="utf-8") as f:
    step0_metrics = json.load(f)

with (
    DATA_DIR / "eval_step_000050_metrics.json"
).open(encoding="utf-8") as f:
    step50_metrics = json.load(f)

def load_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]

eval_records = load_jsonl(BASELINE_EVAL_PATH)

step0_predictions = {
    r["id"]: r
    for r in load_jsonl(
        DATA_DIR / "baseline_step0_predictions_12.jsonl"
    )
}

step50_predictions = {
    r["id"]: r
    for r in load_jsonl(
        DATA_DIR / "eval_step_000050_predictions_12.jsonl"
    )
}

assert len(eval_records) == 12

expected_ids = {r["id"] for r in eval_records}

assert set(step0_predictions) == expected_ids
assert set(step50_predictions) == expected_ids

# ==========================================
# 2. INFERENCE ON THE FIXED VALIDATION SET
# ==========================================

model.eval()

results = []
references = []
predictions = []

preview_audio = None

try:
    with torch.inference_mode():

        for index, record in enumerate(
            tqdm(eval_records, desc="Step-200 evaluation")
        ):
            audio_path = (
                EVAL_AUDIO_DIR / f"{record['id']}.mp3"
            )

            assert audio_path.exists(), audio_path

            decoder = AudioDecoder(
                str(audio_path),
                sample_rate=SAMPLE_RATE,
                num_channels=1
            )

            samples = decoder.get_all_samples()

            waveform = (
                samples.data[0]
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            if index == 0:
                preview_audio = waveform.copy()

            inputs = processor.feature_extractor(
                waveform,
                sampling_rate=SAMPLE_RATE,
                return_attention_mask=True,
                return_tensors="pt"
            )

            inputs = {
                name: tensor.to(DEVICE)
                for name, tensor in inputs.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(**inputs).logits

            assert torch.isfinite(logits).all()

            predicted_ids = logits.argmax(dim=-1)

            predicted_text = processor.batch_decode(
                predicted_ids.cpu()
            )[0]

            predicted_text = normalize_amharic_ctc(
                predicted_text
            )

            reference = record["transcription"]

            blank_fraction = (
                (predicted_ids == tokenizer.pad_token_id)
                .float()
                .mean()
                .item()
            )

            result = {
                "id": record["id"],
                "speaker_id": record["speaker_id"],
                "reference": reference,
                "prediction": predicted_text,
                "cer": cer(reference, predicted_text),
                "wer": wer(reference, predicted_text),
                "blank_fraction": blank_fraction,
                "prediction_length": len(predicted_text)
            }

            results.append(result)
            references.append(reference)
            predictions.append(predicted_text)

            del inputs, logits, predicted_ids

finally:
    # Keep the existing model ready for training.
    model.train()

# ==========================================
# 3. CALCULATE EVALUATION METRICS
# ==========================================

step200_metrics = {
    "training_step": EVAL_STEP,
    "evaluation_recordings": len(results),
    "cer": cer(references, predictions),
    "wer": wer(references, predictions),
    "mean_blank_fraction": float(
        np.mean([r["blank_fraction"] for r in results])
    ),
    "empty_predictions": sum(
        not prediction for prediction in predictions
    ),
    "mean_prediction_length": float(
        np.mean([len(p) for p in predictions])
    )
}

# ==========================================
# 4. SAVE RESULTS
# ==========================================

PREDICTIONS_PATH = (
    DATA_DIR / "eval_step_000200_predictions_12.jsonl"
)

METRICS_PATH = (
    DATA_DIR / "eval_step_000200_metrics.json"
)

with PREDICTIONS_PATH.open("w", encoding="utf-8") as f:
    for result in results:
        f.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

with METRICS_PATH.open("w", encoding="utf-8") as f:
    json.dump(step200_metrics, f, indent=2)

# ==========================================
# 5. THREE-WAY COMPARISON
# ==========================================

all_metrics = [
    ("Step 0", step0_metrics),
    ("Step 50", step50_metrics),
    ("Step 200", step200_metrics)
]

print("\n" + "=" * 72)
print("AMHARIC ASR: THREE-CHECKPOINT COMPARISON")
print("=" * 72)

print(
    f"{'Checkpoint':<15}"
    f"{'CER':>12}"
    f"{'WER':>12}"
    f"{'Blank %':>14}"
    f"{'Empty':>10}"
)

print("-" * 65)

for name, m in all_metrics:
    print(
        f"{name:<15}"
        f"{m['cer'] * 100:>11.2f}%"
        f"{m['wer'] * 100:>11.2f}%"
        f"{m['mean_blank_fraction'] * 100:>13.2f}%"
        f"{m['empty_predictions']:>10}"
    )

print("\nCER CHANGE")
print(
    "Step 0 -> 200:",
    round(
        (step200_metrics["cer"] - step0_metrics["cer"]) * 100,
        2
    ),
    "percentage points"
)

print(
    "Step 50 -> 200:",
    round(
        (step200_metrics["cer"] - step50_metrics["cer"]) * 100,
        2
    ),
    "percentage points"
)

# ==========================================
# 6. COMPARE ACTUAL AMHARIC PREDICTIONS
# ==========================================

print("\n" + "=" * 72)
print("TRANSCRIPTIONS: BEFORE VS AFTER")
print("=" * 72)

for index in [0, 5, 11]:
    current = results[index]
    record_id = current["id"]

    previous0 = step0_predictions[record_id]
    previous50 = step50_predictions[record_id]

    print(f"\n{'-' * 70}")
    print(f"RECORDING {index + 1}")
    print("REFERENCE:")
    print(current["reference"][:220])

    print("\nSTEP 0:")
    print(previous0["prediction"][:220] or "[EMPTY]")

    print("\nSTEP 50:")
    print(previous50["prediction"][:220] or "[EMPTY]")

    print("\nSTEP 200:")
    print(current["prediction"][:220] or "[EMPTY]")

    print(
        "\nCER:",
        f"{previous0['cer'] * 100:.1f}%",
        "->",
        f"{previous50['cer'] * 100:.1f}%",
        "->",
        f"{current['cer'] * 100:.1f}%"
    )

print("\nAUDIO FOR RECORDING 1:")
display(Audio(preview_audio, rate=SAMPLE_RATE))

print("\nPredictions saved:", PREDICTIONS_PATH)
print("Metrics saved:", METRICS_PATH)
print("\nSTEP-200 EVALUATION COMPLETE")


## Original source cell 29

Position 30 in the original notebook; original code comments may use different numbering.


In [ ]:

import torch
import numpy as np
from pathlib import Path
from torchcodec.decoders import AudioDecoder

assert successful_updates == 200

BLANK_ID = tokenizer.pad_token_id

# Deterministic training groups.
seen_indices = training_order[::199][:4]
unseen_indices = training_order[800::199][:4]

assert len(seen_indices) == 4
assert len(unseen_indices) == 4

# Build three groups of examples.
groups = {
    "TRAIN: SEEN": [
        train_dataset[i] for i in seen_indices
    ],
    "TRAIN: UNSEEN": [
        train_dataset[i] for i in unseen_indices
    ],
    "VALIDATION": []
}

for record in eval_records[:4]:
    audio_path = EVAL_AUDIO_DIR / f"{record['id']}.mp3"

    decoder = AudioDecoder(
        str(audio_path),
        sample_rate=SAMPLE_RATE,
        num_channels=1
    )
    decoded = decoder.get_all_samples()

    waveform = (
        decoded.data[0].cpu().numpy().astype(np.float32)
    )

    features = processor.feature_extractor(
        waveform,
        sampling_rate=SAMPLE_RATE
    )

    labels = tokenizer.encode(
        record["transcription"],
        add_special_tokens=False
    )

    groups["VALIDATION"].append({
        "input_values": features.input_values[0],
        "labels": labels,
        "id": record["id"]
    })

# Inspect predictions and probabilities.
model.eval()

try:
    with torch.inference_mode():

        for group_name, examples in groups.items():

            losses = []
            blank_argmax_rates = []
            blank_probabilities = []
            nonblank_probabilities = []

            print("\n" + "=" * 65)
            print(group_name)
            print("=" * 65)

            for example in examples:
                batch = ctc_collator([example])

                batch = {
                    key: value.to(DEVICE)
                    for key, value in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    outputs = model(**batch)

                loss = float(outputs.loss.item())

                # Use FP32 for probability diagnostics.
                probs = outputs.logits[0].float().softmax(
                    dim=-1
                )

                frame_predictions = probs.argmax(dim=-1)

                blank_argmax = (
                    frame_predictions == BLANK_ID
                ).float().mean().item()

                blank_mean = (
                    probs[:, BLANK_ID].mean().item()
                )

                # Strongest nonblank probability per frame.
                nonblank = probs.clone()
                nonblank[:, BLANK_ID] = 0.0

                strongest_nonblank = (
                    nonblank.max(dim=-1).values
                )

                mean_nonblank = (
                    strongest_nonblank.mean().item()
                )

                peak_nonblank = (
                    strongest_nonblank.max().item()
                )

                predicted = processor.batch_decode(
                    frame_predictions.unsqueeze(0).cpu()
                )[0]

                # Identify the most probable nonblank tokens,
                # averaged over all acoustic frames.
                mean_token_probs = nonblank.mean(dim=0)
                top_ids = torch.topk(
                    mean_token_probs, k=5
                ).indices.tolist()

                top_tokens = [
                    tokenizer.convert_ids_to_tokens(i)
                    for i in top_ids
                ]

                losses.append(loss)
                blank_argmax_rates.append(blank_argmax)
                blank_probabilities.append(blank_mean)
                nonblank_probabilities.append(mean_nonblank)

                print(f"\nID: {example['id']}")
                print("CTC loss:", round(loss, 4))
                print(
                    "Blank argmax frames:",
                    f"{blank_argmax * 100:.2f}%"
                )
                print(
                    "Mean blank probability:",
                    f"{blank_mean * 100:.2f}%"
                )
                print(
                    "Mean strongest nonblank probability:",
                    f"{mean_nonblank * 100:.2f}%"
                )
                print(
                    "Peak nonblank probability:",
                    f"{peak_nonblank * 100:.2f}%"
                )
                print("Top nonblank tokens:", top_tokens)
                print(
                    "Greedy prediction:",
                    predicted[:100] or "[EMPTY]"
                )

                del batch, outputs, probs, nonblank

            print("\nGROUP SUMMARY")
            print(
                "Mean CTC loss:",
                round(float(np.mean(losses)), 4)
            )
            print(
                "Mean blank argmax:",
                f"{np.mean(blank_argmax_rates) * 100:.2f}%"
            )
            print(
                "Mean blank probability:",
                f"{np.mean(blank_probabilities) * 100:.2f}%"
            )
            print(
                "Mean strongest nonblank probability:",
                f"{np.mean(nonblank_probabilities) * 100:.2f}%"
            )

finally:
    model.train()

print("\nBLANK-DOMINANCE DIAGNOSTIC COMPLETE")
print("No weights were updated.")


## Original source cell 30

Position 31 in the original notebook; original code comments may use different numbering.


In [ ]:

import os
import json
import time
import shutil
import hashlib
import random
import gc
from pathlib import Path

import numpy as np
import torch

# ==========================================
# 1. CONFIGURATION
# ==========================================

START_STEP = 200
END_STEP = 500
SAVE_STEPS = {350, 500}
ACCUMULATION = 4

assert successful_updates == START_STEP
assert scheduler.last_epoch == START_STEP
assert model.is_gradient_checkpointing
assert len(train_dataset) == 5083

# Verify the exact training order.
training_order = list(range(len(train_dataset)))
random.Random(TRAINING_SEED).shuffle(training_order)

with ORDER_PATH.open(encoding="utf-8") as f:
    saved_order = json.load(f)

assert [
    train_records[i]["id"] for i in training_order
] == saved_order["recording_ids"]

assert END_STEP * ACCUMULATION <= len(training_order)

for step in SAVE_STEPS:
    destination = CHECKPOINT_DIR / f"step_{step:06d}"
    assert not destination.exists(), (
        f"Checkpoint already exists: {destination}"
    )

# ==========================================
# 2. VERIFIED CHECKPOINT FUNCTION
# ==========================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def save_training_checkpoint(step, logs):

    name = f"step_{step:06d}"

    local_dir = Path(f"/content/{name}_training_temp")
    partial_dir = CHECKPOINT_DIR / f"{name}.partial"
    final_dir = CHECKPOINT_DIR / name

    for path in [local_dir, partial_dir, final_dir]:
        assert not path.exists(), f"Path exists: {path}"

    local_dir.mkdir(parents=True)

    # Model and tokenizer.
    model.save_pretrained(
        str(local_dir),
        safe_serialization=True
    )

    processor.save_pretrained(str(local_dir))

    # Complete optimizer and training state.
    state = {
        "global_step": step,
        "epoch": 0,
        "next_sample_offset": step * ACCUMULATION,
        "seed": TRAINING_SEED,
        "num_epochs": NUM_EPOCHS,
        "gradient_accumulation": ACCUMULATION,
        "total_steps": TOTAL_STEPS,
        "warmup_steps": WARMUP_STEPS,
        "unfrozen_layers": 2,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_states": torch.cuda.get_rng_state_all(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state()
    }

    torch.save(
        state,
        local_dir / "trainer_state.pt"
    )

    summary = {
        "global_step": step,
        "recordings_processed": step * ACCUMULATION,
        "recent_mean_loss": float(
            np.mean([x["loss"] for x in logs[-25:]])
        ),
        "chunk_log": logs
    }

    with (
        local_dir / "training_summary.json"
    ).open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    # Copy to Drive and verify contents.
    shutil.copytree(local_dir, partial_dir)

    for source in local_dir.iterdir():
        if source.is_file():
            copied = partial_dir / source.name

            assert copied.exists()
            assert sha256_file(source) == sha256_file(copied), (
                f"Checksum mismatch: {source.name}"
            )

    os.rename(partial_dir, final_dir)

    (final_dir / "COMPLETE").write_text(
        "Verified checkpoint.",
        encoding="utf-8"
    )

    shutil.rmtree(local_dir)

    print(f"\nCHECKPOINT SAVED: STEP {step}")
    print("Location:", final_dir)
    print("Verification: PASSED\n")


# ==========================================
# 3. CONTINUE TRAINING
# ==========================================

model.train()
torch.cuda.reset_peak_memory_stats()

training_log = []
start_time = time.perf_counter()

print("TRAINING CONTINUATION")
print("-" * 55)
print("Starting step:", START_STEP)
print("Target step:", END_STEP)
print("Additional recordings:", (END_STEP - START_STEP) * 4)
print("Checkpoint steps:", sorted(SAVE_STEPS))
print("Training order: VERIFIED")
print()

for step in range(START_STEP + 1, END_STEP + 1):

    optimizer.zero_grad(set_to_none=True)

    offset = (step - 1) * ACCUMULATION
    indices = training_order[
        offset:offset + ACCUMULATION
    ]

    assert len(indices) == ACCUMULATION

    losses = []

    for record_index in indices:

        batch = ctc_collator(
            [train_dataset[record_index]]
        )

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite CTC loss at step {step}"
            )

        losses.append(float(loss.item()))

        scaler.scale(
            loss / ACCUMULATION
        ).backward()

        del batch, outputs, loss

    # ======================================
    # 4. GRADIENT CLIPPING + UPDATE
    # ======================================

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        encoder_params + head_params,
        max_norm=MAX_GRAD_NORM,
        error_if_nonfinite=True
    )

    scale_before = scaler.get_scale()

    scaler.step(optimizer)
    scaler.update()

    if scaler.get_scale() < scale_before:
        raise RuntimeError(
            f"AMP skipped optimizer update at step {step}. "
            "Recover from the last verified checkpoint."
        )

    scheduler.step()
    successful_updates = step

    mean_loss = float(np.mean(losses))

    training_log.append({
        "step": step,
        "loss": mean_loss,
        "gradient_norm": float(grad_norm),
        "head_lr": optimizer.param_groups[1]["lr"],
        "encoder_lr": optimizer.param_groups[0]["lr"]
    })

    if step % 25 == 0:

        recent = training_log[-25:]

        recent_loss = float(np.mean([
            entry["loss"] for entry in recent
        ]))

        print(
            f"Step {step:04d}/{TOTAL_STEPS} | "
            f"Last-25 loss: {recent_loss:.4f} | "
            f"Head LR: "
            f"{optimizer.param_groups[1]['lr']:.8f}"
        )

    # ======================================
    # 5. PERIODIC VERIFIED CHECKPOINTS
    # ======================================

    if step in SAVE_STEPS:
        save_training_checkpoint(
            step,
            training_log.copy()
        )

# ==========================================
# 6. FINAL REPORT
# ==========================================

elapsed = time.perf_counter() - start_time

assert successful_updates == END_STEP

print("\n" + "=" * 60)
print("TRAINING COMPLETED THROUGH STEP 500")
print("=" * 60)

print("Current step:", successful_updates)
print("Recordings processed:", END_STEP * ACCUMULATION)

print(
    "Mean loss, steps 201-500:",
    round(
        float(np.mean([
            x["loss"] for x in training_log
        ])),
        4
    )
)

print(
    "Mean loss, last 25 steps:",
    round(
        float(np.mean([
            x["loss"] for x in training_log[-25:]
        ])),
        4
    )
)

print("Elapsed minutes:", round(elapsed / 60, 2))

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

print("\nVERIFIED CHECKPOINTS")

for step in sorted(SAVE_STEPS):
    path = CHECKPOINT_DIR / f"step_{step:06d}"

    print(
        f"Step {step}:",
        "VERIFIED" if (path / "COMPLETE").exists()
        else "MISSING"
    )

print("\nREADY FOR STEP-500 EVALUATION")
